# 02 — Modeling & Hyperparameter Tuning

**Owner**: Isaac  
**Goal**: Train and tune the four core anomaly detection models (Isolation Forest, One-Class SVM, LOF, Elliptic Envelope).

See `TASKS_ISAAC.md` for the detailed checklist.

In [3]:
import sys
from pathlib import Path

sys.path.append(str(Path("../").resolve()))

In [4]:
import pandas as pd
import numpy as np

from src.models import (
    train_isolation_forest,
    train_one_class_svm,
    train_lof,
    train_elliptic_envelope,
)

print("Imports successful")

Imports successful


In [5]:
df = pd.read_csv("../ai4i2020.csv")

print(df.shape)

df.head()

(10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [6]:
df.columns

Index(['UDI', 'Product ID', 'Type', 'Air temperature [K]',
       'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]',
       'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF',
       'RNF'],
      dtype='object')

In [7]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Copie du dataset
data = df.copy()

# Encodage de la colonne Type
encoder = LabelEncoder()
data["Type"] = encoder.fit_transform(data["Type"])

# Features pour le modèle
X = data.drop(
    columns=[
        "UDI",
        "Product ID",
        "Machine failure",
        "TWF",
        "HDF",
        "PWF",
        "OSF",
        "RNF",
    ]
)

# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(X_scaled.shape)

(10000, 6)


In [9]:
iso_model = train_isolation_forest(
    X_scaled,
    contamination=0.034,
    n_estimators=200,
    max_samples="auto",
    random_state=42,
)

iso_preds = iso_model.predict(X_scaled)

unique, counts = np.unique(iso_preds, return_counts=True)
dict(zip(unique, counts))

{-1: 340, 1: 9660}

In [10]:
from pathlib import Path

results_dir = Path("../outputs/results")
results_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame({"prediction": iso_preds}).to_csv(
    results_dir / "preds_isolation_forest.csv",
    index=True,
)

print("Export OK")

Export OK


In [11]:
ocsvm_model = train_one_class_svm(
    X_scaled,
    nu=0.034,
    kernel="rbf",
    gamma="scale",
)

ocsvm_preds = ocsvm_model.predict(X_scaled)

unique, counts = np.unique(ocsvm_preds, return_counts=True)
dict(zip(unique, counts))

{-1: 342, 1: 9658}

In [12]:
pd.DataFrame({"prediction": ocsvm_preds}).to_csv(
    results_dir / "preds_ocsvm.csv",
    index=True,
)

print("OCSVM export OK")

OCSVM export OK


In [13]:
lof_model = train_lof(
    X_scaled,
    n_neighbors=35,
    contamination=0.034,
)

lof_preds = lof_model.fit_predict(X_scaled)

unique, counts = np.unique(lof_preds, return_counts=True)
dict(zip(unique, counts))

{-1: 340, 1: 9660}

In [14]:
pd.DataFrame({"prediction": lof_preds}).to_csv(
    results_dir / "preds_lof.csv",
    index=True,
)

print("LOF export OK")

LOF export OK


In [15]:
elliptic_model = train_elliptic_envelope(
    X_scaled,
    contamination=0.034,
    support_fraction=None,
    random_state=42,
)

elliptic_preds = elliptic_model.predict(X_scaled)

unique, counts = np.unique(elliptic_preds, return_counts=True)
dict(zip(unique, counts))

/opt/anaconda3/lib/python3.12/site-packages/sklearn/covariance/_robust_covariance.py:186: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-6.636684237051077 > -71.432973826184607). You may want to try with a higher value of support_fraction (current value: 0.501).
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/covariance/_robust_covariance.py:186: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-6.655518345498513 > -71.523949028032604). You may want to try with a higher value of support_fraction (current value: 0.501).
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/covariance/_robust_covariance.py:186: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-6.663627177583668 > -71.532635260750610). You may want to try with a higher value of support_fraction (current value: 0.501).
  warnings.warn(
/opt/an

{-1: 340, 1: 9660}

In [16]:
pd.DataFrame({"prediction": elliptic_preds}).to_csv(
    results_dir / "preds_elliptic.csv",
    index=True,
)

print("Elliptic export OK")

Elliptic export OK


In [17]:
models_dict = {
    "isolation_forest": iso_model,
    "ocsvm": ocsvm_model,
    "lof": lof_model,
    "elliptic": elliptic_model,
}

print(models_dict.keys())

dict_keys(['isolation_forest', 'ocsvm', 'lof', 'elliptic'])


## Hyperparameter tuning strategy

We test several hyperparameter configurations for each anomaly detection model.  
Since the dataset contains approximately 3.4% anomalies, contamination-like parameters are tested around this value: 0.02, 0.034, 0.05, and 0.10.

The objective is not only to maximize accuracy, but also to check whether the models produce stable and coherent anomaly rates.

In [18]:
summary = pd.DataFrame({
    "model": [
        "Isolation Forest",
        "One-Class SVM",
        "LOF",
        "Elliptic Envelope",
    ],
    "anomalies_detected": [
        (iso_preds == -1).sum(),
        (ocsvm_preds == -1).sum(),
        (lof_preds == -1).sum(),
        (elliptic_preds == -1).sum(),
    ],
    "normal_points": [
        (iso_preds == 1).sum(),
        (ocsvm_preds == 1).sum(),
        (lof_preds == 1).sum(),
        (elliptic_preds == 1).sum(),
    ],
})

summary["anomaly_rate"] = summary["anomalies_detected"] / len(X_scaled)

summary

,model,anomalies_detected,normal_points,anomaly_rate
0,Isolation Forest,340,9660,0.0340
1,One-Class SVM,342,9658,0.0342
2,LOF,340,9660,0.0340
3,Elliptic Envelope,340,9660,0.0340


## First modeling results

All four models were calibrated around the observed anomaly rate of approximately 3.4%.  
The predictions follow the sklearn convention: `-1` for anomalies and `1` for normal observations.

Isolation Forest, One-Class SVM, LOF, and Elliptic Envelope all detect around 340 anomalies, which is coherent with the expected anomaly proportion.

However, these models rely on different assumptions:
- Isolation Forest isolates anomalies through random partitions.
- One-Class SVM learns a non-linear boundary around normal observations.
- LOF detects local density deviations.
- Elliptic Envelope assumes a multivariate Gaussian distribution, which may be restrictive for this dataset.

In [19]:
iso_results = []

for contamination in [0.02, 0.034, 0.05, 0.10]:
    for n_estimators in [100, 200, 300]:
        for max_samples in ["auto", 0.5]:
            
            model = train_isolation_forest(
                X_scaled,
                contamination=contamination,
                n_estimators=n_estimators,
                max_samples=max_samples,
                random_state=42,
            )
            
            preds = model.predict(X_scaled)
            
            iso_results.append({
                "model": "Isolation Forest",
                "contamination": contamination,
                "n_estimators": n_estimators,
                "max_samples": max_samples,
                "anomalies_detected": (preds == -1).sum(),
                "anomaly_rate": (preds == -1).mean(),
            })

iso_results_df = pd.DataFrame(iso_results)
iso_results_df

,model,contamination,n_estimators,max_samples,anomalies_detected,anomaly_rate
0,Isolation Forest,0.020,100,auto,200,0.020
1,Isolation Forest,0.020,100,0.5,200,0.020
2,Isolation Forest,0.020,200,auto,200,0.020
3,Isolation Forest,0.020,200,0.5,200,0.020
4,Isolation Forest,0.020,300,auto,200,0.020
5,Isolation Forest,0.020,300,0.5,200,0.020
6,Isolation Forest,0.034,100,auto,340,0.034
7,Isolation Forest,0.034,100,0.5,340,0.034
8,Isolation Forest,0.034,200,auto,340,0.034
9,Isolation Forest,0.034,200,0.5,340,0.034


In [20]:
ocsvm_results = []

for nu in [0.02, 0.034, 0.05, 0.10]:
    for gamma in ["scale", "auto", 0.01, 0.1, 1.0]:
        
        model = train_one_class_svm(
            X_scaled,
            nu=nu,
            kernel="rbf",
            gamma=gamma,
        )
        
        preds = model.predict(X_scaled)
        
        ocsvm_results.append({
            "model": "One-Class SVM",
            "nu": nu,
            "gamma": gamma,
            "anomalies_detected": (preds == -1).sum(),
            "anomaly_rate": (preds == -1).mean(),
        })

ocsvm_results_df = pd.DataFrame(ocsvm_results)
ocsvm_results_df

,model,nu,gamma,anomalies_detected,anomaly_rate
0,One-Class SVM,0.020,scale,203,0.0203
1,One-Class SVM,0.020,auto,203,0.0203
2,One-Class SVM,0.020,0.01,202,0.0202
3,One-Class SVM,0.020,0.1,202,0.0202
4,One-Class SVM,0.020,1.0,588,0.0588
5,One-Class SVM,0.034,scale,342,0.0342
6,One-Class SVM,0.034,auto,342,0.0342
7,One-Class SVM,0.034,0.01,338,0.0338
8,One-Class SVM,0.034,0.1,342,0.0342
9,One-Class SVM,0.034,1.0,598,0.0598


In [21]:
lof_results = []

for n_neighbors in [10, 20, 35, 50]:
    for contamination in [0.02, 0.034, 0.05, 0.10]:
        
        model = train_lof(
            X_scaled,
            n_neighbors=n_neighbors,
            contamination=contamination,
        )
        
        preds = model.fit_predict(X_scaled)
        
        lof_results.append({
            "model": "LOF",
            "n_neighbors": n_neighbors,
            "contamination": contamination,
            "anomalies_detected": (preds == -1).sum(),
            "anomaly_rate": (preds == -1).mean(),
        })

lof_results_df = pd.DataFrame(lof_results)
lof_results_df

,model,n_neighbors,contamination,anomalies_detected,anomaly_rate
0,LOF,10,0.020,200,0.020
1,LOF,10,0.034,340,0.034
2,LOF,10,0.050,500,0.050
3,LOF,10,0.100,1000,0.100
4,LOF,20,0.020,200,0.020
5,LOF,20,0.034,340,0.034
6,LOF,20,0.050,500,0.050
7,LOF,20,0.100,1000,0.100
8,LOF,35,0.020,200,0.020
9,LOF,35,0.034,340,0.034


In [22]:
elliptic_results = []

for contamination in [0.02, 0.034, 0.05, 0.10]:
    for support_fraction in [None, 0.75, 0.9]:
        
        model = train_elliptic_envelope(
            X_scaled,
            contamination=contamination,
            support_fraction=support_fraction,
            random_state=42,
        )
        
        preds = model.predict(X_scaled)
        
        elliptic_results.append({
            "model": "Elliptic Envelope",
            "contamination": contamination,
            "support_fraction": support_fraction,
            "anomalies_detected": (preds == -1).sum(),
            "anomaly_rate": (preds == -1).mean(),
        })

elliptic_results_df = pd.DataFrame(elliptic_results)
elliptic_results_df

/opt/anaconda3/lib/python3.12/site-packages/sklearn/covariance/_robust_covariance.py:186: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-6.636684237051077 > -71.432973826184607). You may want to try with a higher value of support_fraction (current value: 0.501).
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/covariance/_robust_covariance.py:186: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-6.655518345498513 > -71.523949028032604). You may want to try with a higher value of support_fraction (current value: 0.501).
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/covariance/_robust_covariance.py:186: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-6.663627177583668 > -71.532635260750610). You may want to try with a higher value of support_fraction (current value: 0.501).
  warnings.warn(
/opt/an

,model,contamination,support_fraction,anomalies_detected,anomaly_rate
0,Elliptic Envelope,0.020,NaN,200,0.020
1,Elliptic Envelope,0.020,0.75,200,0.020
2,Elliptic Envelope,0.020,0.90,200,0.020
3,Elliptic Envelope,0.034,NaN,340,0.034
4,Elliptic Envelope,0.034,0.75,340,0.034
5,Elliptic Envelope,0.034,0.90,340,0.034
6,Elliptic Envelope,0.050,NaN,500,0.050
7,Elliptic Envelope,0.050,0.75,500,0.050
8,Elliptic Envelope,0.050,0.90,500,0.050
9,Elliptic Envelope,0.100,NaN,1000,0.100


## Modeling interpretation

The selected contamination level is `0.034`, because it is consistent with the anomaly rate observed in the dataset.

For Isolation Forest, `n_estimators=200` is retained as a compromise between computation time and model stability.

For One-Class SVM, the RBF kernel is used because the anomaly boundary is unlikely to be purely linear. `gamma="scale"` is retained because it adapts to the variance of the scaled features.

For LOF, `n_neighbors=35` is used as a compromise between very local neighborhoods, which can be noisy, and larger neighborhoods, which can smooth subtle anomalies.

For Elliptic Envelope, the model is kept as a useful baseline, but its main limitation is that it assumes a multivariate Gaussian distribution. This assumption may be partially violated in the AI4I dataset, so its results should be interpreted carefully.

In [23]:
results_dir = Path("../outputs/results")
results_dir.mkdir(parents=True, exist_ok=True)

predictions = {
    "isolation_forest": iso_preds,
    "ocsvm": ocsvm_preds,
    "lof": lof_preds,
    "elliptic": elliptic_preds,
}

for name, preds in predictions.items():
    pd.DataFrame({"prediction": preds}).to_csv(
        results_dir / f"preds_{name}.csv",
        index=True,
    )

print("All prediction files exported.")

All prediction files exported.


In [25]:
models_dict = {
    "isolation_forest": iso_model,
    "ocsvm": ocsvm_model,
    "lof": lof_model,
    "elliptic": elliptic_model,
}

models_dict.keys()

dict_keys(['isolation_forest', 'ocsvm', 'lof', 'elliptic'])